In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.19 Addition of Angular Momenta and Clebsch–Gordan Coefficients

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.19",
    title="Addition of Angular Momenta and Clebsch–Gordan Coefficients",
    blurb="Two spins, one total. When a system carries two angular momenta — an orbit "
    "and a spin, or two spins — what is conserved is their sum, and the states "
    "reorganize into multiplets of definite total angular momentum. The coefficients "
    "that make the change of basis are famous for filling tables in the back of "
    "textbooks; we will never look one up. They are the components of eigenvectors, "
    "and the computer hands them to us by diagonalizing or by climbing a ladder.",
    difficulty="advanced",
    estimate="165–200 min",
)

## Notebook overview

The previous notebook ended on an open problem. Spin–orbit coupling contributes an energy $\propto\mathbf L\cdot
\mathbf S$, and to evaluate it we needed the eigenstates of $J^2=(\mathbf L+\mathbf S)^2$ — we needed to
know how to **add** two angular momenta. That is this notebook, and it is a tool we will use again and
again: any time a system carries two angular momenta (two spins, a spin and an orbit, two orbits), the
individual pieces can trade angular momentum back and forth, but their **total** $\mathbf J=\mathbf J_1+
\mathbf J_2$ is conserved and labels the states.

The structure is rigid and beautiful. A product space of dimension $(2j_1+1)(2j_2+1)$ reorganizes into
multiplets of definite total $j$ running from $|j_1-j_2|$ to $j_1+j_2$ (the **triangle rule**), each
appearing exactly once, each of dimension $2j+1$ — and the dimensions add up. There are two natural
bases: the **uncoupled** basis $|j_1,m_1\rangle|j_2,m_2\rangle$, in which each part is sharp, and the
**coupled** basis $|j,m\rangle$, in which the total is sharp. The **Clebsch–Gordan coefficients** are the
overlaps $\langle j_1 m_1 j_2 m_2|j m\rangle$ that transform between them.

In a traditional course this is where one learns to read tables of Clebsch–Gordan coefficients out of the
back of a book. We invert that entirely, in the spirit of the whole volume: **there is no table** — the
coupled states are simply the eigenvectors of $J^2$ and $J_z$, and the coefficients are their components
in the uncoupled basis, handed to us by the computer. We do it two ways and watch them agree: by
**diagonalizing** $J^2$ in the product space (fast, general), and by the **ladder construction** (start
from the top state and lower, the [§6.12](harmonic-oscillator.ipynb)/[§6.14](angular-momentum-algebra.ipynb) method again), which fixes the standard Condon–Shortley phase
convention.

Two couplings carry all the physics we need. Two spin-$\tfrac12$'s combine into a **singlet** ($j=0$,
antisymmetric) and a **triplet** ($j=1$, symmetric) — and the singlet is *a* maximally-entangled Bell state, like the $(|00\rangle+|11\rangle)/\sqrt2$ of [§6.8](bloch-sphere-entanglement.ipynb),
so angular-momentum coupling and entanglement turn out to be the same mathematics, while the exchange
symmetry (symmetric triplet, antisymmetric singlet) is the seed of the Pauli principle that the next
notebook makes into law. And an orbital $l$ coupled to spin $\tfrac12$ gives $j=l\pm\tfrac12$, in which
$\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ is finally **diagonal** — resolving the open problem of [§6.18](spin-magnetic.ipynb) and setting
up the fine structure of [§6.21](perturbation-fine-structure.ipynb).

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — the [§6.14](angular-momentum-algebra.ipynb) `angular_momentum_matrices`, `numpy.kron` to build
$\mathbf J_1\otimes I+I\otimes\mathbf J_2$, `numpy.linalg.eigh` to diagonalize $J^2$, the total lowering
operator $J_-=J_{1-}+J_{2-}$ for the ladder, and the Clebsch–Gordan coefficients read off as eigenvector
components.

> **Conventions.** $\hbar=1$. The uncoupled product basis is ordered by `numpy.kron`: index $i_1(2j_2+1)
> +i_2$ with $m_1=j_1-i_1$, $m_2=j_2-i_2$ (descending $m$). We fix each coupled state's overall phase by a
> **Condon–Shortley**-style convention (its first significant component is positive real); the ladder
> construction produces this convention naturally, and we align the diagonalization to it. **Scope:** we
> cover the decomposition rule, the two construction methods, the coefficients, and the two anchor cases
> ($\tfrac12\otimes\tfrac12$ and $l\otimes\tfrac12$); the **Wigner–Eckart theorem**, the **3j/6j symbols**,
> **tensor operators**, and coupling three or more angular momenta are named as horizons, not developed.
> See Sakurai & Napolitano (§3.8) and Griffiths; and Notebooks [§6.14](angular-momentum-algebra.ipynb) (the matrices/ladder), [§6.8](bloch-sphere-entanglement.ipynb)
> (entanglement/Bell), [§6.18](spin-magnetic.ipynb) (spin, the $\mathbf L\cdot\mathbf S$ question), [§6.6](pauli-uncertainty.ipynb) (compatible observables).

## Theory in brief

### Why total angular momentum

For two parts with angular momenta $\mathbf J_1,\mathbf J_2$, the individual components are generally not
conserved (they exchange angular momentum, e.g. through $\mathbf L\cdot\mathbf S$), but the **total** is:

```{math}
:label: eq-total-J
\mathbf J=\mathbf J_1+\mathbf J_2=\mathbf J_1\otimes I+I\otimes\mathbf J_2 ,
```

which generates rotations of the whole system. So the good quantum numbers are those of $J^2$ and $J_z$
(built from the [§6.14](angular-momentum-algebra.ipynb) matrices with `numpy.kron`).

### The two bases and the decomposition rule

The **uncoupled** basis $|j_1,m_1\rangle\otimes|j_2,m_2\rangle$ diagonalizes $J_1^2,J_2^2,J_{1z},J_{2z}$
(each part sharp); the **coupled** basis $|j_1,j_2;j,m\rangle$ diagonalizes $J_1^2,J_2^2,J^2,J_z$ (the
total sharp). Both span the same product space, and it decomposes as

```{math}
:label: eq-decomposition
j_1\otimes j_2=\bigoplus_{j=|j_1-j_2|}^{j_1+j_2} j,\qquad \sum_{j=|j_1-j_2|}^{j_1+j_2}(2j+1)=(2j_1+1)(2j_2+1) ,
```

the **triangle rule**: each total $j$ from $|j_1-j_2|$ to $j_1+j_2$ appears once, and since $J_z$
eigenvalues add ($m=m_1+m_2$), counting how many uncoupled states have each $m$ pins down which
multiplets appear.

### The Clebsch–Gordan coefficients

The coupled states expand in the uncoupled basis,

```{math}
:label: eq-cg
|j,m\rangle=\sum_{m_1+m_2=m}\langle j_1 m_1 j_2 m_2|j m\rangle\,|j_1,m_1\rangle|j_2,m_2\rangle ,
```

and the coefficients $\langle j_1 m_1 j_2 m_2|j m\rangle$ are the **Clebsch–Gordan coefficients** — the
entries of the unitary change-of-basis matrix, i.e. the **components of the coupled eigenvectors** in the
uncoupled basis. No table is needed.

### Method 1 — diagonalize $J^2$

The coupled states are *defined* as joint eigenvectors of $J^2$ and $J_z$, so the first method is the
most direct one imaginable: pose the eigenvalue problem in the uncoupled product basis and let the
computer solve it, schematically

```{math}
:label: eq-cg-diagonalize
J^2=(\mathbf J_1+\mathbf J_2)^2\ \xrightarrow{\ \texttt{numpy.linalg.eigh}\ }\ \{|j,m\rangle\},\qquad \text{CG}=\langle\text{uncoupled}|\text{coupled}\rangle .
```

Build $J^2$ in the product space and diagonalize (simultaneously with $J_z$); group the eigenvectors by
their $J^2$ eigenvalue $j(j+1)$ and $J_z$ eigenvalue $m$; their uncoupled components are the coefficients.

### Method 2 — the ladder construction

The second method trades the eigensolver for the angular-momentum algebra itself: the same lowering
operator that walked down each multiplet in [§6.12](harmonic-oscillator.ipynb) and [§6.14](angular-momentum-algebra.ipynb) also walks down the coupled multiplets,
once a coupled state is in hand to start from (§3.8 of Sakurai & Napolitano gives the general
construction). Its two ingredients are

```{math}
:label: eq-cg-ladder
|j_1+j_2,\,j_1+j_2\rangle=|j_1,j_1\rangle|j_2,j_2\rangle,\qquad J_-=J_{1-}+J_{2-} ,
```

the **stretched** state is unique (highest $m$); apply $J_-$ to descend its ladder, and the next
multiplet's top state is the combination at that $m$ **orthogonal** to what is already built. Repeating
constructs every coupled state and reproduces the Condon–Shortley convention — the [§6.12](harmonic-oscillator.ipynb)/[§6.14](angular-momentum-algebra.ipynb) ladder, now
for coupling. We show it agrees with Method 1.

### The two anchor cases

The smallest nontrivial coupling, two spin-$\tfrac12$'s, already carries most of the physics we need.
The triangle rule allows exactly $j=1$ and $j=0$ (dimensions $3+1=4$), and either method above
delivers their states:

```{math}
:label: eq-singlet-triplet
\tfrac12\otimes\tfrac12=1\oplus0:\quad \underbrace{|1,0\rangle=\tfrac{|{\uparrow\downarrow}\rangle+|{\downarrow\uparrow}\rangle}{\sqrt2}}_{\text{triplet, symmetric}},\quad \underbrace{|0,0\rangle=\tfrac{|{\uparrow\downarrow}\rangle-|{\downarrow\uparrow}\rangle}{\sqrt2}}_{\text{singlet, antisymmetric = Bell}} ,
```

the triplet ($J^2=2$) symmetric, the singlet ($J^2=0$) antisymmetric and *a* maximally-entangled Bell
state, like the $(|00\rangle+|11\rangle)/\sqrt2$ of [§6.8](bloch-sphere-entanglement.ipynb). And for an orbital coupled to spin (expanding $J^2=(\mathbf L+\mathbf S)^2$ gives the identity),

```{math}
:label: eq-spin-orbit-coupled
l\otimes\tfrac12=(l+\tfrac12)\oplus(l-\tfrac12),\qquad \mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)=\tfrac{\hbar^2}{2}\big[j(j+1)-l(l+1)-s(s+1)\big] ,
```

diagonal in the coupled basis — the $\mathbf L\cdot\mathbf S$ of [§6.18](spin-magnetic.ipynb), now evaluated, and the
fine-structure splitting of [§6.21](perturbation-fine-structure.ipynb).

## Setup

Data and instruments only: the series palette, the single-$j$ matrices $J_x,J_y,J_z,J^2,J_\pm$
built from scratch in [§6.14](angular-momentum-algebra.ipynb) and restated here, the phase
convention that fixes each coupled state's overall sign, the ordering of the `numpy.kron`
product basis, and the spin–orbit operator $\mathbf L\cdot\mathbf S$ carried over from
[§6.18](spin-magnetic.ipynb). Everything this notebook is named for is deliberately absent:
you write the total operators `total_J` in Exercise 1, the diagonalization route
`couple_by_diagonalization` in Exercise 2, the Clebsch–Gordan matrix `cg_matrix` in
Exercise 3, and the ladder route `couple_by_ladder` in Exercise 4 — and every later exercise
runs on the ones you wrote.

The Setup below holds this notebook's data and instruments — nothing you
are asked to build. It is collapsed so the building stays yours; expand it
whenever you want the details.

<!-- setup-policy: v2 -->

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT  # data: the series palette


# built from scratch in §6.14 (Exercise 1, where the ladder matrix elements
# $\hbar\sqrt{j(j+1)-m(m\pm1)}$ are filled in by hand for integer and half-integer $j$ alike);
# restated here as an instrument — the single-$j$ workbench this notebook couples.
def angular_momentum_matrices(j, hbar=1.0):
    r"""The angular-momentum matrices $J_x,J_y,J_z,J^2,J_+,J_-$ for a given $j$ — reused from §6.14."""
    dim = int(round(2 * j + 1))
    m = np.arange(j, -j - 1, -1.0)
    Jz = hbar * np.diag(m).astype(complex)
    Jp = np.zeros((dim, dim), dtype=complex)
    Jm = np.zeros((dim, dim), dtype=complex)
    for a in range(dim):
        ma = m[a]
        cp = j * (j + 1) - ma * (ma + 1)
        if cp > 1e-12:
            Jp[a - 1, a] = hbar * np.sqrt(cp)
        cm = j * (j + 1) - ma * (ma - 1)
        if cm > 1e-12:
            Jm[a + 1, a] = hbar * np.sqrt(cm)
    Jx = (Jp + Jm) / 2
    Jy = (Jp - Jm) / 2j
    J2 = Jx @ Jx + Jy @ Jy + Jz @ Jz
    return Jx, Jy, Jz, J2, Jp, Jm


# instrument: a sign convention, not a result — it removes the arbitrary overall phase that any
# eigensolver or normalization leaves behind, so that the two construction routes can be compared
# entry by entry. Which phase one settles on teaches nothing about coupling.
def _canonical_phase(v, tol=1e-9):
    """Fix the overall sign so the first significant component is positive (a Condon–Shortley-style convention)."""
    for x in v:
        if abs(x) > tol:
            return v if x.real > 0 else -v
    return v


# instrument: the basis-ordering convention rather than the physics — it records what the rows and
# columns of every product-space object mean (index $i_1(2j_2+1)+i_2$, both $m$'s descending), a
# lookup table and not a result.
def uncoupled_labels(j1, j2):
    """The $(m_1,m_2)$ labels of the ``numpy.kron`` product basis, in order."""
    m1 = np.arange(j1, -j1 - 1, -1.0)
    m2 = np.arange(j2, -j2 - 1, -1.0)
    return [(m1[i // len(m2)], m2[i % len(m2)]) for i in range(len(m1) * len(m2))]


# instrument: the $\mathbf L\cdot\mathbf S$ operator of §6.18, restated here. Summing three tensor
# products is not this notebook's lesson; the lesson of Exercise 6 is that the operator so built
# turns out *diagonal* in the coupled basis, with eigenvalue $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$.
def spin_orbit_operator(l, s=0.5):
    r"""The spin–orbit operator $\mathbf L\cdot\mathbf S=L_xS_x+L_yS_y+L_zS_z$ on the $l\otimes s$ space {eq}`eq-spin-orbit-coupled`."""
    Lx, Ly, Lz, *_ = angular_momentum_matrices(l)
    sx, sy, sz, *_ = angular_momentum_matrices(s)
    return np.kron(Lx, sx) + np.kron(Ly, sy) + np.kron(Lz, sz)

## Exercise 1 — Total angular momentum in the product space

Adding two angular momenta is, first of all, an act of arithmetic on **operators that live in
different spaces**. $\mathbf J_1$ acts on the $(2j_1+1)$-dimensional space of the first part and
$\mathbf J_2$ on the $(2j_2+1)$-dimensional space of the second, so before they can be added each
must be extended to the product space by padding it with the *other's* identity — that is what
$\mathbf J=\mathbf J_1\otimes I+I\otimes\mathbf J_2$ {eq}`eq-total-J` means, and `numpy.kron` is
the padding. Everything downstream is built on these operators, so this is the notebook's first
and most consequential construction. The physics it is meant to expose is the one that makes the
whole subject necessary: $J^2$ commutes with $J_z$, so the *total* has sharp quantum numbers,
but it does **not** commute with $J_{1z}$, so the parts do not {eq}`eq-decomposition`.

1. Write `total_J(j1, j2)`, returning $J_x,J_y,J_z,J^2,J_-$ on the $(2j_1+1)(2j_2+1)$-dimensional
   product space: take the Setup's [§6.14](angular-momentum-algebra.ipynb)
   `angular_momentum_matrices` for each part, pad each with the other's identity using
   `numpy.kron`, add the pairs, and form $J^2=J_x^2+J_y^2+J_z^2$ by matrix multiplication.
   **Write this one yourself** — the implementation is the lesson.
2. Build the total operators for, say, $j_1=1$ and $j_2=\tfrac12$; note that the uncoupled basis
   diagonalizes $J_{1z},J_{2z}$ while the coupled basis diagonalizes $J^2,J_z$.
3. Compute $[J^2,J_z]$ and $[J^2,J_{1z}]$, padding $J_{1z}$ alone with the second part's identity.
4. Confirm the first vanishes and the second does not — $m_1$ is not a good quantum number for the
   total. The total is conserved; the parts are not.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.check(
    total_ok,
    "J²=(J₁+J₂)² commutes with Jz but not with J₁z — the total angular momentum, not its parts, labels the coupled states",
)

## Exercise 2 — The decomposition rule and dimension counting

The coupled states are *defined* as joint eigenvectors of $J^2$ and $J_z$, so the most direct
route to them is to pose that eigenvalue problem in the product basis and let the computer solve
it {eq}`eq-cg-diagonalize`. Two practicalities make it work. First, $J^2$ alone is massively
degenerate — every state of a multiplet shares its eigenvalue — so diagonalizing $J^2+c\,J_z$
with an *irrational* $c$ (say $1/\pi$) lifts the degeneracy without disturbing the eigenvectors:
the two operators commute, so they share eigenvectors, and no two combinations $j(j+1)+cm$ can
collide by accident. Second, an eigenvector's labels are recovered from its own expectation
values, $\langle J^2\rangle=j(j+1)$ inverted by the quadratic formula
$j=\tfrac12(-1+\sqrt{1+4\langle J^2\rangle})$ and $\langle J_z\rangle=m$. Once the multiplets are
in hand, their bookkeeping is the **triangle rule** $j=|j_1-j_2|,\dots,j_1+j_2$ with
$\sum(2j+1)=(2j_1+1)(2j_2+1)$ {eq}`eq-decomposition` — a counting identity the coupling must
satisfy exactly, and the acceptance test for the construction.

1. Write `couple_by_diagonalization(j1, j2)`, returning a dictionary of coupled states keyed by
   $(j,m)$: build the total operators with your Exercise 1 `total_J`, diagonalize $J^2+J_z/\pi$
   with `numpy.linalg.eigh`, label each eigenvector by the expectation values above (rounded to
   the nearest half-integer), and fix its overall sign with the Setup's `_canonical_phase`.
   **Write this one yourself** — the implementation is the lesson.
2. For several $(j_1,j_2)$ — say $\tfrac12\otimes\tfrac12$, $1\otimes\tfrac12$, $1\otimes1$,
   $\tfrac32\otimes\tfrac12$ — couple with it and read off the distinct $j$ values.
3. Confirm they run $|j_1-j_2|$ to $j_1+j_2$ in integer steps.
4. Confirm each appears once with dimension $2j+1$.
5. Confirm the dimensions sum to $(2j_1+1)(2j_2+1)$ — the triangle rule, verified.

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    decomp_ok,
    "j₁⊗j₂ decomposes into multiplets |j₁−j₂| through j₁+j₂ (each once, dim 2j+1), and the dimensions sum to (2j₁+1)(2j₂+1)",
)

## Exercise 3 — Clebsch–Gordan coefficients by diagonalization

The coupled states of Exercise 2 came out as vectors in the uncoupled basis, and their components
already *are* the Clebsch–Gordan coefficients $\langle j_1 m_1 j_2 m_2|j m\rangle$ {eq}`eq-cg`:
collected into one array — rows the coupled states $|j,m\rangle$, columns the uncoupled products
$|m_1,m_2\rangle$ — they form the change-of-basis matrix $U$ between two orthonormal descriptions
of the same space. That is what the tables in the back of textbooks tabulate, and it is why $U$
must come out **unitary**, $U^\dagger U=I$: an honest check on the whole construction, since
nothing in Exercise 2 imposed orthonormality across different multiplets. For $1\otimes\tfrac12$
the famous entries are $\sqrt{2/3}$ and $\sqrt{1/3}$, whose squares $2/3$ and $1/3$ are the
probabilities of finding each uncoupled product in the coupled state.

1. Couple $1\otimes\tfrac12$ with the `couple_by_diagonalization` you wrote in Exercise 2, which
   groups the eigenvectors by $j(j+1)$ and $m$ as it labels them.
2. Write `cg_matrix(states, j1, j2)`: sort the coupled keys by descending $(j,m)$, stack each
   state's (real) uncoupled components as a row of a matrix $U$, and return $U$ together with the
   coupled key list and the Setup's `uncoupled_labels` for the columns.
3. Read off one coupled state's expansion, say $|\tfrac32,\tfrac12\rangle$ — those components are
   the CG coefficients, computed and not looked up.
4. Verify that $U$ is unitary ($U^\dagger U=I$). The coefficients are eigenvector components — no
   table.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.check(
    cg_diag_ok,
    "the coupled eigenvectors' uncoupled components form a unitary matrix — Clebsch–Gordan coefficients are the overlaps ⟨uncoupled|coupled⟩",
)

In [ ]:
# (solution hidden on the public site)


## Exercise 4 — Clebsch–Gordan coefficients by the ladder construction

The eigensolver is not the only way down, and it is not the historical one. The same lowering
operator that walked down a single multiplet in [§6.12](harmonic-oscillator.ipynb) and
[§6.14](angular-momentum-algebra.ipynb) walks down the coupled multiplets too, and it needs only
one state to start from {eq}`eq-cg-ladder`. That state is free: the **stretched** state
$|j_1+j_2,j_1+j_2\rangle=|j_1,j_1\rangle|j_2,j_2\rangle$ is the unique product with the maximum
$m=j_1+j_2$, so it can be nothing but the top of the top multiplet. Applying
$J_-=J_{1-}+J_{2-}$ and renormalizing walks that multiplet to its bottom; the next multiplet's
top state is then the vector at $m=j_1+j_2-1$ **orthogonal** to the one already built, obtained
by projecting a product basis vector against what exists (Gram–Schmidt) — and the construction
repeats until the multiplets run out. Because the sign of each step is fixed by the operator and
not by a solver, this route produces the standard Condon–Shortley convention by itself; the
diagonalization of Exercise 2 was aligned to it by hand. The two must therefore agree entry for
entry, which is how each certifies the other.

1. Write `couple_by_ladder(j1, j2)`, returning the same $(j,m)$-keyed dictionary as your
   Exercise 2 `couple_by_diagonalization`: for each $j$ from $j_1+j_2$ down to $|j_1-j_2|$, seed
   the multiplet with the first product basis vector at $m=j$ (from the Setup's
   `uncoupled_labels`) that survives projection against the states already built, normalize it and
   fix its phase with `_canonical_phase`, then descend with the $J_-$ your Exercise 1 `total_J`
   returns, normalizing at every rung. **Write this one yourself** — the implementation is the
   lesson.
2. Build the coupled states of $1\otimes\tfrac12$ both ways.
3. Compare them state by state, taking the largest norm of the difference.
4. Confirm the two constructions agree to machine precision. The ladder builds the coupling and
   fixes the convention — the [§6.12](harmonic-oscillator.ipynb)/[§6.14](angular-momentum-algebra.ipynb) method again.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    ladder_ok,
    "the ladder construction reproduces the coupled basis and its Condon–Shortley phase convention — it matches the diagonalization",
)

## Exercise 5 — Singlet and triplet: two spin-$\tfrac12$'s

The smallest nontrivial coupling is also the most consequential. Two spin-$\tfrac12$'s span four
product states, and the triangle rule allows only $j=1$ and $j=0$, of dimensions $3$ and $1$
{eq}`eq-singlet-triplet`. The two aligned products pass through untouched, but the two mixed ones
recombine into a symmetric and an antisymmetric pair: the symmetric **triplet** ($J^2=2$) and the
antisymmetric **singlet** ($J^2=0$). Two things follow at once. The singlet
$(|{\uparrow\downarrow}\rangle-|{\downarrow\uparrow}\rangle)/\sqrt2$ is *a* maximally-entangled
Bell state, a sibling of the $(|00\rangle+|11\rangle)/\sqrt2$ of
[§6.8](bloch-sphere-entanglement.ipynb) — angular-momentum coupling and entanglement are the same
mathematics wearing different clothes. And the exchange symmetry visible here, symmetric for the
triplet and antisymmetric for the singlet, is the seed of the Pauli principle that
[§6.20](identical-particles.ipynb) makes into law.

1. Form $\tfrac12\otimes\tfrac12$ with the `couple_by_ladder` you wrote in Exercise 4.
2. Identify the triplet and singlet states among the four.
3. Verify their $J^2$ eigenvalues (2 for the triplet, 0 for the singlet).
4. Read each state's uncoupled components and note which are symmetric and which antisymmetric
   under exchanging the two spins.

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    np.array([J2_triplet, J2_singlet]),
    np.array([2.0, 0.0]),
    "two spin-½'s couple into a symmetric triplet (j=1, J²=2) and an antisymmetric singlet (j=0, J²=0 — the Bell state)",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — Spin–orbit coupling evaluated

[§6.18](spin-magnetic.ipynb) could write the spin–orbit interaction down but not evaluate it: in
the uncoupled basis $\mathbf L\cdot\mathbf S$ is a mess of off-diagonal elements, because
$L_xS_x+L_yS_y$ flips $m_l$ and $m_s$ in opposite directions. The coupled basis dissolves the
problem. Squaring $\mathbf J=\mathbf L+\mathbf S$ gives
$\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ {eq}`eq-spin-orbit-coupled`, and all three
operators on the right are diagonal in the coupled basis at once — so $\mathbf L\cdot\mathbf S$
is diagonal there too, with eigenvalue $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$ depending on nothing but
the three quantum numbers. Coupling $l$ with $s=\tfrac12$ yields $j=l\pm\tfrac12$, so a single
level splits into two, one raised and one lowered, with multiplicities $2j+1$. That splitting is
the **fine structure** of atomic spectra, whose physical size [§6.21](perturbation-fine-structure.ipynb)
computes by perturbation theory.

1. Couple $l=1$ with $s=\tfrac12$ using your Exercise 2 `couple_by_diagonalization`, obtaining the
   $j=\tfrac32$ and $j=\tfrac12$ multiplets.
2. Build $\mathbf L\cdot\mathbf S$ on that space with the Setup's `spin_orbit_operator`.
3. Take a representative state of each multiplet and confirm $\langle\mathbf L\cdot\mathbf S\rangle$
   matches the predicted $\tfrac12[j(j+1)-l(l+1)-s(s+1)]$.
4. Report the two values and their multiplicities $2j+1$ — the $\mathbf L\cdot\mathbf S$ of
   [§6.18](spin-magnetic.ipynb), now computed.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    np.array(ls_grid),
    np.array(ls_pred),
    "L·S is diagonal in the coupled basis with eigenvalue ½[j(j+1)−l(l+1)−s(s+1)] — the spin–orbit interaction, evaluated",
    rtol=1e-9,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — A coupling of your choice, both ways *(student)*

Nothing in the two constructions was special to $1\otimes\tfrac12$, and the way to be sure of
that is to run them on a case neither was demonstrated on. Two $p$-electrons couple their orbital
angular momenta as $1\otimes1=2\oplus1\oplus0$ {eq}`eq-decomposition` — a nine-dimensional product
space carrying a $D$, a $P$ and an $S$ term, the multiplet structure behind atomic term symbols —
and $\tfrac32\otimes\tfrac12=2\oplus1$ is an equally good alternative. Whichever you take, the
coupled and uncoupled descriptions remain two orthonormal bases of one space, so the full
Clebsch–Gordan matrix must come out unitary {eq}`eq-cg` and the two methods must agree: that,
and nothing more, is the entire content of "addition of angular momenta".

1. Pick a coupling $(j_1,j_2)$ — worked below as $1\otimes1$.
2. Build the coupled states both ways, with your Exercise 2 `couple_by_diagonalization` and your
   Exercise 4 `couple_by_ladder`, and list the multiplets.
3. Confirm the two constructions agree state by state.
4. Assemble the CG matrix with your Exercise 3 `cg_matrix` and confirm it is unitary.
5. Check the dimensions: $\sum(2j+1)=(2j_1+1)(2j_2+1)$. The method is general — any coupling, no
   tables.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.check(
    student_ok,
    "both methods give the same multiplets and (unitary) CG coefficients for 1⊗1 = 2⊕1⊕0 — the coupled basis by diagonalization or ladder, for any j₁⊗j₂",
)

## Exercise 8 — The algebra of combination *(synthesis)*

When two angular momenta join, their sum is what is conserved, and the states rearrange themselves into
multiplets of definite total — a rule so rigid that just counting how many states carry each $J_z$ value
tells you which multiplets appear. The famous Clebsch–Gordan coefficients that carry out the
rearrangement are nothing but the components of eigenvectors, and we found them by diagonalizing and by
climbing ladders, never by opening a table.

There is no new computation here: the machinery is the result. Two couplings paid
immediate dividends. Two spins gave the **singlet and triplet** — the entanglement of Movement I wearing
the clothes of angular momentum, and the exchange symmetry that the next notebook will make into a law.
And an orbit coupled to a spin let us finally **evaluate $\mathbf L\cdot\mathbf S$**, the fine-structure
interaction that [§6.18](spin-magnetic.ipynb) could only name. The next notebook ([§6.20](identical-particles.ipynb)) takes the singlet/triplet exchange
symmetry seriously as a principle: when the two particles are **identical**, nature allows only certain
symmetries under their exchange — bosons symmetric, fermions antisymmetric — and that single rule, the
Pauli exclusion principle, builds atoms, fills the $2n^2$ shells, and gives matter its stability.

The coefficients in the back of the textbook always looked like something handed down. They are not —
they are what falls out of a matrix when you ask it for its eigenvectors, and every one of them is a
statement about how much of one arrangement is contained in another.

## Notebook summary

The addition of angular momenta — the tool that resolves the open problem of [§6.18](spin-magnetic.ipynb) and sets up identical particles.

- **The total is conserved** {eq}`eq-total-J`: $\mathbf J=\mathbf J_1+\mathbf J_2$ (`numpy.kron`);
  $[J^2,J_z]=0$ but $[J^2,J_{1z}]\ne0$, so the total, not the parts, labels the coupled states.
- **The triangle rule** {eq}`eq-decomposition`: $j_1\otimes j_2=\bigoplus_{|j_1-j_2|}^{j_1+j_2}j$, with
  $\sum(2j+1)=(2j_1+1)(2j_2+1)$.
- **Clebsch–Gordan coefficients** {eq}`eq-cg`: the overlaps $\langle\text{uncoupled}|\text{coupled}\rangle$
  — the components of the coupled eigenvectors, obtained by **diagonalizing** $J^2$ (`numpy.linalg.eigh`)
  or by the **ladder** ($J_-=J_{1-}+J_{2-}$); the two methods agree. No table.
- **Singlet and triplet** {eq}`eq-singlet-triplet`: $\tfrac12\otimes\tfrac12=1\oplus0$; the singlet is a
  Bell state (maximally entangled, like the $(|00\rangle+|11\rangle)/\sqrt2$ of [§6.8](bloch-sphere-entanglement.ipynb)), coupling = entanglement, exchange symmetry seeds the Pauli principle ([§6.20](identical-particles.ipynb)).
- **Spin–orbit** {eq}`eq-spin-orbit-coupled`: $l\otimes\tfrac12=(l+\tfrac12)\oplus(l-\tfrac12)$, with
  $\mathbf L\cdot\mathbf S=\tfrac12(J^2-L^2-S^2)$ diagonal — the operator of [§6.18](spin-magnetic.ipynb) evaluated, fine structure ([§6.21](perturbation-fine-structure.ipynb)).

The Clebsch–Gordan coefficients are eigenvector components, not table entries. Next, the exchange symmetry
glimpsed here becomes the law of identical particles.

## Outlook

- **Identical particles and the symmetrization postulate ([§6.20](identical-particles.ipynb))**: the singlet/triplet exchange symmetry
  made a law — bosons symmetric, fermions antisymmetric, the Pauli exclusion principle.
- **Fine structure ([§6.21](perturbation-fine-structure.ipynb))**: $\mathbf L\cdot\mathbf S$ and the relativistic corrections by perturbation
  theory; the sodium D-line doublet, the anomalous Zeeman effect.
- **The Wigner–Eckart theorem, the 3j/6j symbols, tensor operators, and coupling three or more angular
  momenta** (horizons, named — the systematic machinery of angular-momentum coupling).
- **Cross-reference** [§6.14](angular-momentum-algebra.ipynb) (the angular-momentum matrices/ladder), [§6.8](bloch-sphere-entanglement.ipynb) (entanglement/the Bell state), [§6.18](spin-magnetic.ipynb)
  (spin, the $\mathbf L\cdot\mathbf S$ question), [§6.6](pauli-uncertainty.ipynb) (compatible observables), and forward to [§6.20](identical-particles.ipynb), [§6.21](perturbation-fine-structure.ipynb).

In [ ]:
from ecp.style import footer

footer()